# Exercise 2.1 — Car Detection and Tracking (Main Street)

This notebook implements moving-car detection and tracking using:
- **Frame differencing** (between consecutive grayscale frames)
- **Background subtraction** (MOG2)

Only the **main street** area is processed through a Region of Interest (ROI).

In [1]:
import cv2
import numpy as np
from pathlib import Path


class CentroidTracker:
    def __init__(self, max_disappeared=12, max_distance=55):
        self.next_object_id = 0
        self.objects = {}
        self.disappeared = {}
        self.max_disappeared = max_disappeared
        self.max_distance = max_distance

    def _register(self, centroid):
        self.objects[self.next_object_id] = centroid
        self.disappeared[self.next_object_id] = 0
        self.next_object_id += 1

    def _deregister(self, object_id):
        del self.objects[object_id]
        del self.disappeared[object_id]

    def update(self, rects):
        if len(rects) == 0:
            for object_id in list(self.disappeared.keys()):
                self.disappeared[object_id] += 1
                if self.disappeared[object_id] > self.max_disappeared:
                    self._deregister(object_id)
            return self.objects

        input_centroids = np.zeros((len(rects), 2), dtype=np.int32)
        for i, (x, y, w, h) in enumerate(rects):
            input_centroids[i] = (x + w // 2, y + h // 2)

        if len(self.objects) == 0:
            for c in input_centroids:
                self._register(c)
            return self.objects

        object_ids = list(self.objects.keys())
        object_centroids = np.array(list(self.objects.values()))

        dists = np.linalg.norm(
            object_centroids[:, np.newaxis, :] - input_centroids[np.newaxis, :, :], axis=2
        )

        rows = dists.min(axis=1).argsort()
        cols = dists.argmin(axis=1)[rows]

        used_rows = set()
        used_cols = set()

        for row, col in zip(rows, cols):
            if row in used_rows or col in used_cols:
                continue
            if dists[row, col] > self.max_distance:
                continue

            object_id = object_ids[row]
            self.objects[object_id] = input_centroids[col]
            self.disappeared[object_id] = 0
            used_rows.add(row)
            used_cols.add(col)

        unused_rows = set(range(dists.shape[0])) - used_rows
        unused_cols = set(range(dists.shape[1])) - used_cols

        for row in unused_rows:
            object_id = object_ids[row]
            self.disappeared[object_id] += 1
            if self.disappeared[object_id] > self.max_disappeared:
                self._deregister(object_id)

        for col in unused_cols:
            self._register(input_centroids[col])

        return self.objects

In [2]:
# Optional quick check before running full detector
video_test = Path("Traffic_Laramie_1.mp4")
cap_test = cv2.VideoCapture(str(video_test))
ok_test, frame_test = cap_test.read()
print("Video opened:", cap_test.isOpened())
print("First frame read:", ok_test)
if ok_test:
    print("Frame shape:", frame_test.shape)
cap_test.release()

Video opened: True
First frame read: True
Frame shape: (600, 1040, 3)


In [3]:
# --- Video path (same folder as notebook) ---
video_path = Path("Traffic_Laramie_1.mp4")

cap = cv2.VideoCapture(str(video_path))
if not cap.isOpened():
    raise FileNotFoundError(f"Could not open video: {video_path.resolve()}")

# Read first frame for ROI setup
ok, first_frame = cap.read()
if not ok:
    cap.release()
    raise RuntimeError("Could not read first frame from video.")

h, w = first_frame.shape[:2]

# Main-street ROI (lower part of frame)
# You can tune these values if needed for your exact expected output.
x0 = int(0.03 * w)
y0 = int(0.58 * h)
x1 = int(0.97 * w)
y1 = int(0.98 * h)

prev_gray = cv2.cvtColor(first_frame, cv2.COLOR_BGR2GRAY)
prev_gray = cv2.GaussianBlur(prev_gray, (5, 5), 0)

bg_sub = cv2.createBackgroundSubtractorMOG2(history=500, varThreshold=35, detectShadows=False)
tracker = CentroidTracker(max_disappeared=12, max_distance=60)
seen_ids = set()

while True:
    ret, frame = cap.read()
    if not ret:
        break

    display = frame.copy()

    # ROI mask
    roi_mask = np.zeros((h, w), dtype=np.uint8)
    roi_mask[y0:y1, x0:x1] = 255

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gray = cv2.GaussianBlur(gray, (5, 5), 0)

    # 1) Frame differencing
    diff = cv2.absdiff(gray, prev_gray)
    _, diff_bin = cv2.threshold(diff, 25, 255, cv2.THRESH_BINARY)

    # 2) Background subtraction
    fg = bg_sub.apply(frame)
    _, fg_bin = cv2.threshold(fg, 200, 255, cv2.THRESH_BINARY)

    # Combine both methods, then constrain to ROI
    motion = cv2.bitwise_or(diff_bin, fg_bin)
    motion = cv2.bitwise_and(motion, motion, mask=roi_mask)

    # Morphological cleanup
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))
    motion = cv2.morphologyEx(motion, cv2.MORPH_OPEN, kernel, iterations=1)
    motion = cv2.morphologyEx(motion, cv2.MORPH_CLOSE, kernel, iterations=2)
    motion = cv2.dilate(motion, kernel, iterations=1)

    contours, _ = cv2.findContours(motion, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    detections = []
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area < 700:
            continue

        x, y, bw, bh = cv2.boundingRect(cnt)

        # Basic vehicle-like shape filter
        if bw < 28 or bh < 20:
            continue

        detections.append((x, y, bw, bh))

    objects = tracker.update(detections)

    # Draw ROI
    cv2.rectangle(display, (x0, y0), (x1, y1), (0, 0, 255), 2)
    cv2.putText(display, "Main street", (x0 + 10, max(30, y0 - 10)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)

    # Draw detections
    for (x, y, bw, bh) in detections:
        cv2.rectangle(display, (x, y), (x + bw, y + bh), (0, 255, 0), 2)

    # Draw tracking IDs
    for object_id, centroid in objects.items():
        cx, cy = centroid
        seen_ids.add(object_id)
        cv2.putText(display, f"ID {object_id}", (cx - 20, cy - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 0), 2)
        cv2.circle(display, (cx, cy), 3, (255, 255, 0), -1)

    cv2.putText(display, f"Current tracked: {len(objects)}", (20, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
    cv2.putText(display, f"Unique IDs seen: {len(seen_ids)}", (20, 62),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

    cv2.imshow("Traffic Car Detection and Tracking", display)
    cv2.imshow("Motion mask", motion)

    prev_gray = gray

    # Press q to quit
    key = cv2.waitKey(20) & 0xFF
    if key == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()